# Notebook 45 - Compression headroom on the second corpus (CIC-IoMT-2024)

**Why this notebook.** Notebook 40 measured, on CICIoT2023, where pruning starts to cost deployed risk: with two full epochs of recovery a 90% cut is free on both networks and the scoring rule does not matter, while one cheap pass at 80 to 90% escalates benign traffic and silently removes three rare attack families; the pruned students were also better calibrated than the dense reference in 43 of 50 paired cells. Every one of those results rests on one corpus. This notebook runs the same protocol on CIC-IoMT-2024 through the bridge of notebook 32 (official train side only, exact deduplication, 20% stratified validation carve-out; the official test files are never opened), with the teachers of notebook 32 and the channel scores and frozen 40% structures of notebook 33.

**What it does.** GPU, about three hours. (1) Loads the cached bridge and the two accepted teachers. (2) Rebuilds the five removal orders from the committed score tables, checks that the frozen 40% structures are their prefixes, and calibrates 70, 80 and 90% prefixes by realised MACs. (3) Cheap recovery (one pass over a seeded 10% subset, five seeds) for the dense reference and 5 rules x 4 budgets; full recovery (two epochs, three seeds) for the dense reference and the five rules at 90%. Every model is recalibrated on a fixed training slice and audited on the class-balanced evaluation subsample (HSR, benign escalation, attack miss, family and fine macro-F1, per-family and per-class recall, ECE with its sign) and, after recalibration, on a 200,000-row natural-prior sample of the validation carve-out. (4) Writes the paired analysis, the verdict against the readings fixed in stage 2, and 600 dpi figures.

**Readings, fixed in stage 2 before any result.** Headroom under the cheap pass and after two epochs (NB40's rule: best rule's paired HSR gap at least 0.02 with the interval above zero), rule separation where headroom opens, family loss under the cheap pass and its repair by two epochs, and paired calibration. Three cross-corpus expectations are stated and reported held or not held.

In [ ]:
# Stage 1 - bootstrap (GPU runtime)
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os, sys, json, gc, time, copy, itertools, subprocess, importlib
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn
from scipy import stats
import matplotlib
import matplotlib.pyplot as plt

REPO = Path("/content/drive/MyDrive/SABER_Research/saber-ids")
IOMT_REPO = Path("/content/drive/MyDrive/IOMT_Compression_Research/iomt-compression-research")
os.chdir(REPO)
for p in (str(REPO), str(IOMT_REPO / "src")):
    if p not in sys.path:
        sys.path.insert(0, p)
importlib.invalidate_caches()
from src.saber.bridge_iomt import build_bridge
from src.saber.taxonomy import DEFAULT_COST_PROFILES
from src.saber.metrics import full_model_audit
from src.saber.surgery import prune_cnn1d_channels, profile_forward_flops, count_parameters
from src.saber.postg5 import calibrate_prefix_count

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
R = REPO / "results/saber"
IN32, IN33 = R / "32_iomt_bridge", R / "33_iomt_scores_structures"
OUT = R / "45_iomt_headroom"; OUT.mkdir(parents=True, exist_ok=True)
DATA_CACHE = REPO / "data/iomt_bridge"; MODEL_DIR = REPO / "models/iomt"          # both git-ignored, written by NB32
for f in ("shallow_scores.csv", "deep_scores.csv", "structure_registry.csv"):
    assert (IN33 / f).exists(), f"missing {IN33 / f}; pull the repo"
assert (IN32 / "bridge_manifest.json").exists(), "missing NB32 bridge manifest; pull the repo"
assert (MODEL_DIR / "shallow_teacher_seed0.pt").exists() and (MODEL_DIR / "deep_teacher_seed0.pt").exists(), "NB32 teachers missing under models/iomt"
print("device:", DEVICE, "| torch", torch.__version__)

In [ ]:
# Stage 2 - pre-registration (frozen: rerunning this cell can never change it after its commit)
PREREG = {
    "notebook": "45_iomt_headroom",
    "question": ("on CIC-IoMT-2024, at what realised MAC reduction does a pruned detector, recovered and recalibrated, carry more "
                 "ground-truth risk than the dense model recovered and recalibrated the same way; does the cheap pass lose attack "
                 "families that the aggregate hides; and are the pruned detectors better or worse calibrated than the dense reference"),
    "motivation": ("notebook 40 answered these on CICIoT2023 only (no headroom at 90% after two epochs on either network; a gap "
                   "only at 90% under the cheap pass on the shallow network; rare families lost under the cheap pass; students "
                   "better calibrated in 43 of 50 paired cells). A second corpus with a different feature set, class structure "
                   "and preprocessing is the test of generality"),
    "design": {
        "corpus": ("CIC-IoMT-2024 WiFi/MQTT through the notebook 32 bridge: official train side only, exact duplicates removed, "
                   "seeded 20% stratified validation carve-out, features clipped at the 0.1/99.9 training percentiles and "
                   "standardised; 19 classes in 6 families, 45 features; the official test files are never opened"),
        "teachers": "the notebook 32 accepted teachers (shallow alpha 0.5, deep alpha 0.35), unchanged",
        "scoring_rules": ("random, L1 magnitude, Taylor, Fisher and the graph-weighted leverage score (saber_v2), orders rebuilt "
                          "from the committed notebook 33 score tables with the notebook 33 ordering rule and the per-layer floor "
                          "of 8 channels used on this corpus; the frozen notebook 33 40% structures must be prefixes of these "
                          "orders or the notebook stops"),
        "budgets": "40% (frozen notebook 33 structures), 70%, 80%, 90% realised MAC reduction, prefixes calibrated within 1.5 points",
        "cheap_recovery": ("one pass over a seeded 10% subset of the training rows, batch 1,024, Adam 1e-3, cross-entropy with "
                           "each teacher's own class-weight exponent, as notebook 34; seeds 101, 211, 307, 401, 503; dense "
                           "reference recovered under the same seed and subset"),
        "full_recovery": "two epochs over all training rows, batch 1,024, same optimiser; seeds 101, 211, 307; at 90% only, with the dense reference",
        "recalibration": "every recovered model: BatchNorm statistics re-estimated on the first 51,200 rows of a seed-2026 permutation of the training rows, weights frozen",
        "evaluation": ("primary: class-balanced subsample of the validation carve-out (at most 4,000 rows per class, seed 0, as "
                       "notebooks 33 and 34); secondary, recalibrated models only: a 200,000-row natural-prior sample of the "
                       "validation carve-out (seed 2026) for benign escalation, attack miss, fine macro-F1 and ECE"),
        "endpoints": ("HSR under the balanced profile (primary), benign escalation E, attack miss M, family and fine macro-F1, "
                      "per-family recall (benign, DDoS, DoS, MQTT, recon, spoofing), per-class recall, 15-bin equal-mass ECE and "
                      "its sign (mean confidence minus accuracy); the collapse flag uses the IoMT gap rule of notebook 34 "
                      "(eval-mode E exceeds batch-statistics E by more than 0.10)"),
    },
    "gap_threshold": 0.02,
    "gap_threshold_note": ("the notebook 40 threshold, kept for comparability; on this corpus the within-rule seed range of "
                           "recalibrated HSR at 40% in notebook 34 was 0.003 to 0.024 (median 0.013)"),
    "family_loss_threshold": 0.15,
    "readings": {
        "H1_cheap": ("per network: the smallest budget in {70, 80, 90} at which the best rule (lowest mean recalibrated HSR) has "
                     "mean paired gap >= 0.02 with the 95% paired t interval above zero; 'none' if no budget qualifies"),
        "H1_full_90": "per network: the same condition at 90% after two epochs (three seeds)",
        "H2_rule_separation": ("at each budget where H1_cheap holds: largest paired mean difference between two rules >= 0.02 with "
                               "at least one Holm-corrected paired contrast significant at 0.05"),
        "F_family_loss": ("per network and rule at 90% under the cheap pass: families whose mean recall is more than 0.15 below "
                          "the dense reference's; 'repaired' if after two epochs no family of the four standard rules (random, "
                          "magnitude, Taylor, Fisher) is more than 0.15 below the dense reference; leverage reported separately"),
        "C_calibration": ("paired ECE (student minus dense, same seed) in every cell: counts of cells with the interval entirely "
                          "below zero (student better), entirely above zero (worse) and straddling; the same on the natural-prior "
                          "sample at 90% after two epochs"),
    },
    "cross_corpus_expectations": {
        "X1": "no headroom at 90% after two epochs on either network (H1_full_90 false for both)",
        "X2": "pruned students better calibrated than the dense reference in at least half of the paired cells",
        "X3": "the cheap pass at 90% loses at least one family (deficit > 0.15) on at least one network",
    },
    "reporting_rule": "every reading reported as measured; an expectation that fails changes the paper's generality statement, it is not dropped",
    "seeds_cheap": [101, 211, 307, 401, 503], "seeds_full": [101, 211, 307], "budgets": [0.4, 0.7, 0.8, 0.9],
    "no_test_access": True, "no_new_selection": True,
}
pre_path = OUT / "N45_PREREGISTRATION.json"
if pre_path.exists():
    print("pre-registration already exists; left untouched:", pre_path)
else:
    pre_path.write_text(json.dumps(PREREG, indent=2)); print("pre-registration written:", pre_path)
print(json.dumps(PREREG, indent=2))

ARCHS = ["shallow", "deep"]
METHODS = ["random", "magnitude", "taylor", "fisher", "saber_v2"]
BUDGETS = PREREG["budgets"]; SEEDS = PREREG["seeds_cheap"]; SEEDS_STD = PREREG["seeds_full"]
GAP = PREREG["gap_threshold"]; FAM_TOL = PREREG["family_loss_threshold"]
SUBSET_FRACTION = 0.10; CAL_SEED = 2026; NAT_SEED = 2026; NAT_ROWS = 200_000; TOL = 0.015; STD_EPOCHS = 2; COLLAPSE_GAP = 0.10
MIN_W = 8                     # the per-layer floor of notebooks 33 and 34 on this corpus (both networks)
NAME = {"random": "random", "magnitude": "magnitude", "taylor": "Taylor", "fisher": "Fisher", "saber_v2": "leverage", "dense": "dense"}

In [ ]:
# Stage 2b - commit the pre-registration before any result exists (skipped automatically once results exist)
def _prereg_committed():
    rel = "results/saber/45_iomt_headroom/N45_PREREGISTRATION.json"
    h = subprocess.run(["git", "log", "-1", "--format=%h", "--", rel], capture_output=True, text=True, cwd=REPO).stdout.strip()
    dirty = subprocess.run(["git", "status", "--porcelain", "--", rel], capture_output=True, text=True, cwd=REPO).stdout.strip()
    return bool(h) and not dirty


_results_exist = any(p.is_file() and p.name != "N45_PREREGISTRATION.json" for p in OUT.rglob("*"))
if _results_exist:
    assert _prereg_committed(), ("results exist but the pre-registration is not committed, so it can no longer be committed before any "
                                 "result; stop here and report it")
    print("results already exist in", OUT.name, "and the pre-registration is committed; nothing to do here")
else:
    os.chdir(REPO)
    r = subprocess.run(["python", "tools/commit_cell.py",
                        "NB45 pre-registration, committed before any result: compression headroom on the second corpus (CIC-IoMT-2024, "
                        "NB32 bridge and teachers, NB33 scores and frozen 40% structures). Five rules x 40/70/80/90% realised MACs, cheap "
                        "recovery (one pass over a seeded 10% subset, 5 seeds) with a dense reference recovered the same way, full recovery "
                        "(two epochs, 3 seeds) at 90%, every model recalibrated on the fixed slice; endpoints on the class-balanced "
                        "subsample plus a 200k natural-prior sample; readings H1 cheap/full (gap >= 0.02, CI above zero), H2 rule "
                        "separation, F family loss (deficit > 0.15) and repair, C paired calibration; cross-corpus expectations X1 to X3 "
                        "stated. No test access, no new selection."],
                       capture_output=True, text=True)
    print(r.stdout); print(r.stderr)

In [ ]:
# Stage 3 - bridge, teachers, evaluation samples and helpers (as notebooks 34 and 40)
TRAIN_LOADER, VAL_LOADER, CLASS_NAMES, taxonomy, MANIFEST = build_bridge(DATA_CACHE)
N_CLASSES = len(CLASS_NAMES)
FAM = np.asarray(taxonomy.class_to_family_index); FAMILIES = list(taxonomy.families)
BENIGN_FAM = FAMILIES.index("benign")


class CNN1D(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.conv = nn.Sequential(nn.Conv1d(1, 64, 3, padding=1), nn.ReLU(), nn.BatchNorm1d(64),
                                  nn.Conv1d(64, 128, 3, padding=1), nn.ReLU(), nn.BatchNorm1d(128))
        self.pool = nn.AdaptiveAvgPool1d(1); self.head = nn.Linear(128, n_classes)

    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)
        return self.head(self.pool(self.conv(x.float())).squeeze(-1))


class DeepCNN1D(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        def blk(i, o):
            return [nn.Conv1d(i, o, 3, padding=1), nn.ReLU(), nn.BatchNorm1d(o)]
        self.conv = nn.Sequential(*blk(1, 64), *blk(64, 128), nn.MaxPool1d(2), *blk(128, 128), *blk(128, 256))
        self.pool = nn.AdaptiveAvgPool1d(1); self.head = nn.Linear(256, n_classes)

    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)
        return self.head(self.pool(self.conv(x.float())).squeeze(-1))


ARCH_CLS = {"shallow": CNN1D, "deep": DeepCNN1D}
TEACHERS, TEACHER_ALPHA = {}, {}
for arch, cls in ARCH_CLS.items():
    payload = torch.load(MODEL_DIR / f"{arch}_teacher_seed0.pt", map_location="cpu", weights_only=False)
    m = cls(N_CLASSES); m.load_state_dict(payload["state_dict"]); TEACHERS[arch] = m.to(DEVICE).eval()
    TEACHER_ALPHA[arch] = float(payload["alpha"])

Xv, Yv = VAL_LOADER.dataset.tensors; VAL_Y_ALL = Yv.numpy()
Xt, Yt = TRAIN_LOADER.dataset.tensors; N_TRAIN = len(Xt)
assert N_TRAIN == MANIFEST["n_train"] and len(Xv) == MANIFEST["n_val"], "bridge tensors do not match the manifest"
_rng = np.random.default_rng(0)
_idx = np.concatenate([_rng.permutation(np.where(VAL_Y_ALL == c)[0])[:4000] for c in range(N_CLASSES) if (VAL_Y_ALL == c).sum() > 0])
_idx = np.random.default_rng(12345).permutation(_idx)
EX_X = Xv[_idx].to(DEVICE); EX_Y = VAL_Y_ALL[_idx]; FAM_Y = FAM[EX_Y]
_nat = np.random.default_rng(NAT_SEED).permutation(len(Xv))[:NAT_ROWS]
NAT_X = Xv[_nat].to(DEVICE); NAT_Y = VAL_Y_ALL[_nat]
EXAMPLE_INPUT = Xv[:8].float().to(DEVICE)
COUNTS = np.bincount(Yt.numpy(), minlength=N_CLASSES)


def class_weights(alpha):
    w = np.zeros(N_CLASSES, dtype=np.float64); nz = COUNTS > 0
    w[nz] = 1.0 / np.power(COUNTS[nz], alpha); w[nz] /= w[nz].mean()
    return torch.tensor(w, dtype=torch.float32, device=DEVICE)


CLASS_W = {a: class_weights(TEACHER_ALPHA[a]) for a in ARCHS}
_cal_idx = torch.randperm(N_TRAIN, generator=torch.Generator().manual_seed(CAL_SEED))[:50 * 1024]
CAL_LOADER = torch.utils.data.DataLoader(torch.utils.data.Subset(TRAIN_LOADER.dataset, _cal_idx.tolist()), batch_size=1024, shuffle=False)


def forward_logits(model, X):
    model.eval()
    with torch.no_grad():
        return torch.cat([model(X[i:i + 8192]).cpu() for i in range(0, len(X), 8192)]).numpy()


def forward_logits_batch_stats(model, X):
    # same weights, BatchNorm normalising by each batch's own statistics; buffers saved and restored (notebooks 34, 39, 40)
    saved = {n: (m.running_mean.clone(), m.running_var.clone(), m.momentum, m.num_batches_tracked.clone())
             for n, m in model.named_modules() if isinstance(m, nn.BatchNorm1d) and m.running_mean is not None}
    model.train()
    for n, m in model.named_modules():
        if n in saved: m.momentum = 0.0
    with torch.no_grad():
        out = torch.cat([model(X[i:i + 8192]).cpu() for i in range(0, len(X), 8192)]).numpy()
    for n, m in model.named_modules():
        if n in saved:
            rm, rv, mom, nbt = saved[n]; m.running_mean.copy_(rm); m.running_var.copy_(rv); m.momentum = mom; m.num_batches_tracked.copy_(nbt)
    model.eval(); return out


def softmax_np(z):
    z = z - z.max(axis=1, keepdims=True); e = np.exp(z); return e / e.sum(axis=1, keepdims=True)


def audit_logits(lg, y):
    a = full_model_audit(lg, y, taxonomy, DEFAULT_COST_PROFILES); pr = softmax_np(lg); pred = lg.argmax(axis=1)
    out = {"b2a": float(a["benign_to_attack_rate"]), "a2b": float(a["attack_to_benign_rate"]), "family_f1": float(a["family_macro_f1"]),
           "fine_f1": float(a["fine_macro_f1"]), "hsr_balanced": float(a["hsr_balanced_soc"]), "hsr_miss": float(a["hsr_miss_sensitive"]),
           "hsr_fatigue": float(a["hsr_alert_fatigue"]), "ece": float(a["ece15"]),
           "conf_minus_acc": float(pr.max(axis=1).mean() - (pred == y).mean())}
    pf = FAM[pred]; fy = FAM[y]
    out.update({f"recall_{FAMILIES[f]}": float((pf[fy == f] == f).mean()) for f in range(len(FAMILIES)) if (fy == f).any()})
    out.update({f"recall_class_{CLASS_NAMES[c]}": float((pred[y == c] == c).mean()) for c in range(N_CLASSES) if (y == c).any()})
    return out


def audit(model, mode="eval"):
    lg = forward_logits(model, EX_X) if mode == "eval" else forward_logits_batch_stats(model, EX_X)
    return audit_logits(lg, EX_Y)


def audit_natural(model):
    a = audit_logits(forward_logits(model, NAT_X), NAT_Y)
    return {k: a[k] for k in ("b2a", "a2b", "family_f1", "fine_f1", "hsr_balanced", "ece", "conf_minus_acc")}


def recalibrate_bn(model, n_batches=50):
    for m in model.modules():
        if isinstance(m, nn.BatchNorm1d): m.reset_running_stats(); m.momentum = None
    model.train()
    with torch.no_grad():
        for i, (xb, _) in enumerate(CAL_LOADER):
            if i >= n_batches: break
            model(xb.to(DEVICE))
    for m in model.modules():
        if isinstance(m, nn.BatchNorm1d): m.momentum = 0.1
    model.eval(); return model


def make_minimal_loader(seed):
    sub = torch.randperm(N_TRAIN, generator=torch.Generator().manual_seed(seed))[: int(N_TRAIN * SUBSET_FRACTION)]
    return torch.utils.data.DataLoader(torch.utils.data.Subset(TRAIN_LOADER.dataset, sub.tolist()), batch_size=1024, shuffle=True, generator=torch.Generator().manual_seed(seed))


def make_full_loader(seed):
    return torch.utils.data.DataLoader(TRAIN_LOADER.dataset, batch_size=1024, shuffle=True, generator=torch.Generator().manual_seed(seed))


def recover(model, loader, epochs, arch):
    opt = torch.optim.Adam(model.parameters(), lr=1e-3); lossf = nn.CrossEntropyLoss(weight=CLASS_W[arch])
    for _ in range(epochs):
        model.train()
        for xb, yb in loader:
            opt.zero_grad(); lossf(model(xb.to(DEVICE)), yb.to(DEVICE)).backward(); opt.step()
    return model.eval()


ctx = [{"architecture": a, "model": "teacher, no recovery", **audit(m), **{f"nat_{k}": v for k, v in audit_natural(m).items()}} for a, m in TEACHERS.items()]
pd.DataFrame(ctx).to_csv(OUT / "N45_teachers.csv", index=False)
print(pd.DataFrame(ctx)[["architecture", "model", "hsr_balanced", "b2a", "a2b", "family_f1", "fine_f1", "ece", "conf_minus_acc", "nat_b2a", "nat_ece"]].round(4).to_string(index=False))
print("IoMT ready | train", N_TRAIN, "| balanced eval rows", len(_idx), "| natural-prior rows", len(_nat), "| families", FAMILIES)

In [ ]:
# Stage 4 - structures: removal orders from the committed NB33 score tables; 40% = frozen NB33 structures; 70/80/90% calibrated
SCORES = {a: pd.read_csv(IN33 / f"{a}_scores.csv") for a in ARCHS}
M0 = {a: profile_forward_flops(TEACHERS[a], EXAMPLE_INPUT)["flops_per_item"] for a in ARCHS}
P0 = {a: count_parameters(TEACHERS[a]) for a in ARCHS}


def removal_order(arch, column):
    # the notebook 33 ordering rule, verbatim: ascending score, skipping a group once its layer is at the floor
    s = SCORES[arch]; left = {p: int((s["module_path"] == p).sum()) for p in s["module_path"].unique()}; seq = []
    for r in s.sort_values(column, ascending=True).itertuples():
        if left[r.module_path] - 1 < MIN_W: continue
        left[r.module_path] -= 1; seq.append((str(r.module_path), int(r.channel_index)))
    return seq


def prune_map_of(pairs):
    pm = {}
    for layer, ch in pairs: pm.setdefault(str(layer), []).append(int(ch))
    return {l: sorted(c) for l, c in pm.items()}


def build_inner(arch, pm):
    st, _ = prune_cnn1d_channels(TEACHERS[arch], pm, EXAMPLE_INPUT, minimum_remaining_per_layer=MIN_W); return st.to(DEVICE)


def realised(arch, st):
    return 1.0 - profile_forward_flops(st, EXAMPLE_INPUT)["flops_per_item"] / M0[arch]


def widths(st):
    return "/".join(str(m.out_channels) for _, m in st.named_modules() if isinstance(m, nn.Conv1d))


def removed_file(arch, method, budget):
    return OUT / f"N45_{arch}_{method}_b{int(round(budget * 100))}_removed_groups.csv"


struct_rows, traces = [], []
for arch in ARCHS:
    for method in METHODS:
        order = removal_order(arch, method)
        fr = pd.read_csv(IN33 / f"{arch}_{method}_r40_removed_groups.csv"); frozen_pairs = list(zip(fr.module_path.astype(str), fr.channel_index.astype(int)))
        assert order[:len(frozen_pairs)] == frozen_pairs, f"{arch}/{method}: the frozen 40% structure is not a prefix of the rebuilt order"
        for budget in BUDGETS:
            if np.isclose(budget, 0.40):
                k, source = len(frozen_pairs), "frozen"
            else:
                cal = calibrate_prefix_count(order, target_reduction=budget, tolerance=TOL,
                                             build_model=lambda k, a=arch, o=order: build_inner(a, prune_map_of(o[:k])),
                                             realised_reduction=lambda st, a=arch: realised(a, st))
                k, source = cal.prefix_length, "calibrated"
                traces += [{"architecture": arch, "method": method, "budget": budget, **t} for t in cal.trace]
            st = build_inner(arch, prune_map_of(order[:k]))
            pd.DataFrame([{"module_path": l, "channel_index": c} for l, c in order[:k]]).to_csv(removed_file(arch, method, budget), index=False)
            raw = audit(st); rr = realised(arch, st)
            struct_rows.append({"architecture": arch, "method": method, "budget": budget, "source": source, "k": k, "realised_reduction": rr,
                                "within_tolerance": bool(abs(rr - budget) <= TOL), "widths": widths(st), "parameters": int(count_parameters(st)),
                                "parameter_ratio": count_parameters(st) / P0[arch], **{f"raw_{kk}": v for kk, v in raw.items() if not kk.startswith("recall_class_")}})
            print(f"{arch:7s} {method:9s} {budget:.2f} [{source:10s}] k={k:3d} realised={rr:.4f} widths={widths(st):16s} params={struct_rows[-1]['parameters']:6d} raw HSR={raw['hsr_balanced']:.4f} raw E={raw['b2a']:.3f}")
            del st; gc.collect()
STRUCT = pd.DataFrame(struct_rows); STRUCT.to_csv(OUT / "N45_structures.csv", index=False); pd.DataFrame(traces).to_csv(OUT / "N45_prefix_search.csv", index=False)
assert STRUCT.within_tolerance.all(), "a structure is outside the 1.5-point tolerance:\n" + STRUCT[~STRUCT.within_tolerance].to_string()
print("all", len(STRUCT), "structures within tolerance; dense MACs per flow", {a: int(M0[a]) for a in ARCHS}, "| dense parameters", P0)

In [ ]:
# Stage 5 - cheap recovery grid: dense reference + 5 rules x 4 budgets, 5 seeds, both networks (resumable)
RUNS = OUT / "N45_runs.csv"
rows = pd.read_csv(RUNS).to_dict("records") if RUNS.exists() else []
done = {(r["regime"], r["architecture"], round(float(r["budget"]), 2), r["method"], int(r["seed"])) for r in rows}


def student_for(arch, method, budget):
    if method == "dense":
        return copy.deepcopy(TEACHERS[arch]).to(DEVICE)
    rm = pd.read_csv(removed_file(arch, method, budget))
    return build_inner(arch, prune_map_of(zip(rm.module_path.astype(str), rm.channel_index.astype(int))))


def run_one(regime, arch, budget, method, seed):
    torch.manual_seed(seed); np.random.seed(seed); t0 = time.time()
    model = student_for(arch, method, budget)
    loader = make_minimal_loader(seed) if regime == "minimal" else make_full_loader(seed)
    recover(model, loader, 1 if regime == "minimal" else STD_EPOCHS, arch)
    tr = audit(model); bs = audit(model, "batch"); recalibrate_bn(model); rc = audit(model); nat = audit_natural(model)
    row = {"regime": regime, "architecture": arch, "budget": budget, "method": method, "seed": seed, "parameters": int(count_parameters(model)),
           "batch_b2a": bs["b2a"], "collapse_as_trained": bool(tr["b2a"] - bs["b2a"] > COLLAPSE_GAP), "seconds": round(time.time() - t0, 1)}
    row.update({f"trained_{k}": v for k, v in tr.items()}); row.update({f"recal_{k}": v for k, v in rc.items()}); row.update({f"nat_{k}": v for k, v in nat.items()})
    del model; gc.collect()
    return row


def run_jobs(jobs):
    for job in jobs:
        key = (job[0], job[1], round(float(job[2]), 2), job[3], int(job[4]))
        if key in done: continue
        row = run_one(*job); rows.append(row); done.add(key); pd.DataFrame(rows).to_csv(RUNS, index=False)
        print(f"{job[0]:8s} {job[1]:7s} b={job[2]:.2f} {job[3]:9s} s{job[4]:4d}: recal HSR {row['recal_hsr_balanced']:.4f} E {row['recal_b2a']:.4f} "
              f"M {row['recal_a2b']:.4f} famF1 {row['recal_family_f1']:.3f} ECE {row['recal_ece']:.3f} | nat E {row['nat_b2a']:.4f} ECE {row['nat_ece']:.3f} | {row['seconds']:.0f}s")


JOBS_MIN = ([("minimal", a, 0.0, "dense", s) for a in ARCHS for s in SEEDS]
            + [("minimal", a, b, m, s) for a in ARCHS for b in BUDGETS for m in METHODS for s in SEEDS])
run_jobs(JOBS_MIN)
print("cheap-recovery runs:", sum(r["regime"] == "minimal" for r in rows), "of", len(JOBS_MIN))

In [ ]:
# Stage 6 - full recovery at 90%: dense reference + 5 rules, two epochs, 3 seeds, both networks (resumable)
JOBS_STD = ([("standard", a, 0.0, "dense", s) for a in ARCHS for s in SEEDS_STD]
            + [("standard", a, 0.90, m, s) for a in ARCHS for m in METHODS for s in SEEDS_STD])
run_jobs(JOBS_STD)
print("full-recovery runs:", sum(r["regime"] == "standard" for r in rows), "of", len(JOBS_STD), "| total rows", len(rows))

In [ ]:
# Stage 7 - analysis and verdict (rules exactly as pre-registered in stage 2)
runs = pd.read_csv(RUNS); STRUCT = pd.read_csv(OUT / "N45_structures.csv")
assert len(runs) == len(JOBS_MIN) + len(JOBS_STD), f"expected {len(JOBS_MIN) + len(JOBS_STD)} runs, found {len(runs)}: finish stages 5 and 6 first"
COL = "recal_hsr_balanced"; STANDARD_RULES = ["random", "magnitude", "taylor", "fisher"]


def series(regime, arch, budget, method, col=COL):
    q = runs[(runs.regime == regime) & (runs.architecture == arch) & (runs.method == method)]
    if method != "dense": q = q[np.isclose(q.budget, budget)]
    return q.set_index("seed")[col].sort_index()


def paired(regime, arch, budget, method, col=COL):
    s = series(regime, arch, budget, method, col); d = series(regime, arch, 0.0, "dense", col); common = s.index.intersection(d.index)
    g = (s.loc[common] - d.loc[common]).to_numpy(); n = len(g); hw = stats.t.ppf(0.975, n - 1) * g.std(ddof=1) / np.sqrt(n) if n > 1 else np.nan
    return {"n": int(n), "gap": float(g.mean()), "gap_lo": float(g.mean() - hw), "gap_hi": float(g.mean() + hw), "min_seed_gap": float(g.min())}


def holm(ps):
    order = np.argsort(ps); out = np.empty(len(ps)); running = 0.0
    for rank, i in enumerate(order):
        running = max(running, min(1.0, ps[i] * (len(ps) - rank))); out[i] = running
    return out


def contrasts(regime, arch, budget):
    piv = pd.DataFrame({m: series(regime, arch, budget, m) for m in METHODS}).dropna(); rows_ = []
    for a, b in itertools.combinations(METHODS, 2):
        d = (piv[a] - piv[b]).to_numpy()
        rows_.append({"regime": regime, "architecture": arch, "budget": budget, "contrast": f"{a}-{b}", "mean": float(d.mean()), "p": float(stats.ttest_rel(piv[a], piv[b]).pvalue)})
    ph = holm(np.array([r["p"] for r in rows_]))
    for r, p in zip(rows_, ph): r["p_holm"] = float(p); r["significant_holm"] = bool(p < 0.05)
    return rows_


FAM_COLS = [f"recal_recall_{f}" for f in FAMILIES]
METRICS = ["recal_hsr_balanced", "recal_b2a", "recal_a2b", "recal_family_f1", "recal_fine_f1", "recal_ece", "recal_conf_minus_acc", "trained_hsr_balanced", "trained_b2a",
           "nat_b2a", "nat_a2b", "nat_fine_f1", "nat_ece", "nat_conf_minus_acc"] + FAM_COLS
summary = []
for regime, budgets in [("minimal", BUDGETS), ("standard", [0.90])]:
    for arch in ARCHS:
        d = runs[(runs.regime == regime) & (runs.architecture == arch) & (runs.method == "dense")]
        summary.append({"regime": regime, "architecture": arch, "budget": 0.0, "method": "dense", "n": len(d), "hsr_sd": float(d[COL].std(ddof=1)), **{m: float(d[m].mean()) for m in METRICS}})
        for b in budgets:
            for m in METHODS:
                q = runs[(runs.regime == regime) & (runs.architecture == arch) & (runs.method == m) & np.isclose(runs.budget, b)]
                e = paired(regime, arch, b, m, "recal_ece"); ne = paired(regime, arch, b, m, "nat_ece")
                defic = {f: float(d[f"recal_recall_{f}"].mean() - q[f"recal_recall_{f}"].mean()) for f in FAMILIES}
                summary.append({"regime": regime, "architecture": arch, "budget": b, "method": m, "n": len(q), "hsr_sd": float(q[COL].std(ddof=1)),
                                **{mm: float(q[mm].mean()) for mm in METRICS}, **paired(regime, arch, b, m),
                                "ece_gap": e["gap"], "ece_gap_lo": e["gap_lo"], "ece_gap_hi": e["gap_hi"],
                                "nat_ece_gap": ne["gap"], "nat_ece_gap_lo": ne["gap_lo"], "nat_ece_gap_hi": ne["gap_hi"],
                                **{f"deficit_{f}": v for f, v in defic.items()}, "max_family_deficit": max(defic.values()), "worst_family": max(defic, key=defic.get)})
SUM = pd.DataFrame(summary); SUM.to_csv(OUT / "N45_summary.csv", index=False)
CON = [c for arch in ARCHS for b in BUDGETS for c in contrasts("minimal", arch, b)] + [c for arch in ARCHS for c in contrasts("standard", arch, 0.90)]
pd.DataFrame(CON).to_csv(OUT / "N45_contrasts.csv", index=False)

H = {}
for arch in ARCHS:
    h1, h1_detail, h2_detail = [], {}, {}
    for b in [x for x in BUDGETS if x > 0.40 + 1e-9]:
        means = {m: float(series("minimal", arch, b, m).mean()) for m in METHODS}; best = min(means, key=means.get); g = paired("minimal", arch, b, best)
        cond = bool(g["gap"] >= GAP and g["gap_lo"] > 0); h1_detail[f"{b:.2f}"] = {"best_rule": best, **g, "condition": cond}
        if cond:
            h1.append(b); cs = [c for c in CON if c["regime"] == "minimal" and c["architecture"] == arch and np.isclose(c["budget"], b)]
            mx = max(abs(c["mean"]) for c in cs); nsig = sum(c["significant_holm"] for c in cs)
            h2_detail[f"{b:.2f}"] = {"max_abs_paired_difference": float(mx), "n_holm_significant": int(nsig), "condition": bool(mx >= GAP and nsig >= 1)}
    means90 = {m: float(series("standard", arch, 0.90, m).mean()) for m in METHODS}; best90 = min(means90, key=means90.get); g90 = paired("standard", arch, 0.90, best90)
    full90 = bool(g90["gap"] >= GAP and g90["gap_lo"] > 0)
    lost = {m: [f for f in FAMILIES if float(SUM[(SUM.regime == "minimal") & (SUM.architecture == arch) & (SUM.method == m) & np.isclose(SUM.budget, 0.90)][f"deficit_{f}"].iloc[0]) > FAM_TOL] for m in METHODS}
    rep = {m: {f: round(float(SUM[(SUM.regime == "standard") & (SUM.architecture == arch) & (SUM.method == m) & np.isclose(SUM.budget, 0.90)][f"deficit_{f}"].iloc[0]), 4) for f in FAMILIES} for m in METHODS}
    repaired = bool(all(v <= FAM_TOL for m in STANDARD_RULES for v in rep[m].values()))
    cells_c = SUM[(SUM.method != "dense") & (SUM.architecture == arch)]
    calib = {"cells": int(len(cells_c)), "student_better": int((cells_c.ece_gap_hi < 0).sum()), "student_worse": int((cells_c.ece_gap_lo > 0).sum()),
             "straddling": int(((cells_c.ece_gap_lo <= 0) & (cells_c.ece_gap_hi >= 0)).sum()),
             "full_90_natural_prior": {r.method: {"nat_ece_gap": round(float(r.nat_ece_gap), 4), "lo": round(float(r.nat_ece_gap_lo), 4), "hi": round(float(r.nat_ece_gap_hi), 4)}
                                       for r in SUM[(SUM.regime == "standard") & (SUM.architecture == arch) & (SUM.method != "dense")].itertuples()},
             "dense_ece_balanced": {reg: round(float(SUM[(SUM.regime == reg) & (SUM.architecture == arch) & (SUM.method == "dense")].recal_ece.iloc[0]), 4) for reg in ("minimal", "standard")},
             "dense_ece_natural": {reg: round(float(SUM[(SUM.regime == reg) & (SUM.architecture == arch) & (SUM.method == "dense")].nat_ece.iloc[0]), 4) for reg in ("minimal", "standard")},
             "dense_conf_minus_acc_balanced": round(float(SUM[(SUM.regime == "standard") & (SUM.architecture == arch) & (SUM.method == "dense")].recal_conf_minus_acc.iloc[0]), 4)}
    e90 = SUM[(SUM.regime == "minimal") & (SUM.architecture == arch) & np.isclose(SUM.budget, 0.90)]; e90f = SUM[(SUM.regime == "standard") & (SUM.architecture == arch) & np.isclose(SUM.budget, 0.90)]
    H[arch] = {"H1_cheap_knee": (min(h1) if h1 else None), "H1_by_budget": h1_detail, "H2_by_budget": h2_detail,
               "H1_full_90": full90, "full_90": {"best_rule": best90, **g90, "means": means90},
               "F_lost_at_90_cheap": lost, "F_deficits_at_90_full": rep, "F_repaired_by_two_epochs_standard_rules": repaired,
               "E_range_90_cheap": [round(float(e90.recal_b2a.min()), 4), round(float(e90.recal_b2a.max()), 4)], "E_dense_cheap": round(float(SUM[(SUM.regime == "minimal") & (SUM.architecture == arch) & (SUM.method == "dense")].recal_b2a.iloc[0]), 4),
               "E_range_90_full": [round(float(e90f.recal_b2a.min()), 4), round(float(e90f.recal_b2a.max()), 4)], "E_dense_full": round(float(SUM[(SUM.regime == "standard") & (SUM.architecture == arch) & (SUM.method == "dense")].recal_b2a.iloc[0]), 4),
               "C_calibration": calib}
cells_all = SUM[SUM.method != "dense"]
X = {"X1_no_headroom_at_90_after_two_epochs_both": bool(not H["shallow"]["H1_full_90"] and not H["deep"]["H1_full_90"]),
     "X2_students_better_calibrated_in_at_least_half": bool((cells_all.ece_gap_hi < 0).sum() >= len(cells_all) / 2),
     "X3_cheap_pass_loses_a_family_on_some_network": bool(any(len(v) > 0 for a in ARCHS for v in H[a]["F_lost_at_90_cheap"].values()))}
verdict = {"notebook": "45_iomt_headroom", "by_architecture": H, "cross_corpus_expectations": X,
           "calibration_all_cells": {"cells": int(len(cells_all)), "student_better": int((cells_all.ece_gap_hi < 0).sum()), "student_worse": int((cells_all.ece_gap_lo > 0).sum())},
           "collapses_as_trained": int(runs.collapse_as_trained.sum()), "runs": int(len(runs)), "prereg": PREREG}
(OUT / "N45_verdict.json").write_text(json.dumps(verdict, indent=2, default=float))

for arch in ARCHS:
    h = H[arch]
    print(f"\n{arch}: H1 cheap knee {h['H1_cheap_knee']} | H1 full 90 {h['H1_full_90']} | families lost at 90 cheap {h['F_lost_at_90_cheap']} | repaired by two epochs {h['F_repaired_by_two_epochs_standard_rules']}")
    for b, v in h["H1_by_budget"].items():
        print(f"  cheap {b}: best {NAME[v['best_rule']]:9s} paired gap {v['gap']:+.4f} [{v['gap_lo']:+.4f}, {v['gap_hi']:+.4f}]")
    v = h["full_90"]; print(f"  full 0.90: best {NAME[v['best_rule']]:9s} paired gap {v['gap']:+.4f} [{v['gap_lo']:+.4f}, {v['gap_hi']:+.4f}] | means {{{', '.join(f'{NAME[m]} {x:.4f}' for m, x in v['means'].items())}}}")
    print(f"  E at 90 cheap {h['E_range_90_cheap']} (dense {h['E_dense_cheap']}) | E at 90 full {h['E_range_90_full']} (dense {h['E_dense_full']})")
    c = h["C_calibration"]; print(f"  calibration: better {c['student_better']} / worse {c['student_worse']} / straddling {c['straddling']} of {c['cells']} cells; dense ECE balanced {c['dense_ece_balanced']} natural {c['dense_ece_natural']} conf-acc {c['dense_conf_minus_acc_balanced']}")
print("\ncross-corpus expectations:", json.dumps(X, indent=1)); print("collapses as trained (gap rule):", verdict["collapses_as_trained"])
show = SUM[["regime", "architecture", "budget", "method", "recal_hsr_balanced", "gap", "gap_lo", "gap_hi", "recal_b2a", "recal_a2b", "recal_family_f1", "recal_ece", "ece_gap", "nat_ece", "max_family_deficit", "worst_family"]]
print(show.round(4).to_string(index=False))

In [ ]:
# Stage 8 - figures (600 dpi, paper names)
SUM = pd.read_csv(OUT / "N45_summary.csv"); runs = pd.read_csv(RUNS)
plt.rcParams.update({"font.family": "serif", "font.serif": ["Liberation Serif", "DejaVu Serif"], "font.size": 7, "axes.labelsize": 7,
                     "xtick.labelsize": 6.5, "ytick.labelsize": 6.5, "legend.fontsize": 6, "pdf.fonttype": 42, "axes.linewidth": 0.6})
COLR = {"random": "#999999", "magnitude": "#E69F00", "taylor": "#56B4E9", "fisher": "#009E73", "saber_v2": "#CC79A7", "dense": "#1F1F1F"}
MARK = {"random": "o", "magnitude": "s", "taylor": "^", "fisher": "D", "saber_v2": "v"}
FAMN = {"benign": "benign", "ddos": "DDoS", "dos": "DoS", "mqtt": "MQTT", "recon": "recon", "spoofing": "spoofing"}

# figure 1: paired HSR gap against budget, cheap pass (lines) and two epochs at 90% (open markers)
fig, axes = plt.subplots(1, 2, figsize=(3.5, 1.9), sharey=True, gridspec_kw={"wspace": 0.08})
for ax, arch in zip(axes, ARCHS):
    ax.axhspan(-GAP, GAP, color="#EEEEEE", zorder=0); ax.axhline(0, color="#B8B8B8", lw=0.6, zorder=1)
    for k, m in enumerate(METHODS):
        g = SUM[(SUM.regime == "minimal") & (SUM.architecture == arch) & (SUM.method == m)].sort_values("budget")
        ax.errorbar(100 * g.budget + (k - 2) * 0.5, g.gap, yerr=[g.gap - g.gap_lo, g.gap_hi - g.gap], color=COLR[m], marker=MARK[m], ms=2.6, lw=0.8, capsize=1.2, label=NAME[m], zorder=3)
        f = SUM[(SUM.regime == "standard") & (SUM.architecture == arch) & (SUM.method == m)]
        if len(f):
            ax.errorbar([97.5 + (k - 2) * 0.5], f.gap, yerr=[f.gap - f.gap_lo, f.gap_hi - f.gap], color=COLR[m], marker=MARK[m], ms=2.6, lw=0.8, capsize=1.2, mfc="white", zorder=3)
    ax.set_title(f"{arch} network", fontsize=7, pad=2); ax.set_xticks([40, 70, 80, 90, 97.5]); ax.set_xticklabels(["40", "70", "80", "90", "90\ntwo ep."])
    ax.set_xlabel("realised reduction (%)"); ax.spines[["top", "right"]].set_visible(False); ax.grid(axis="y", color="#E6E6E6", lw=0.5); ax.set_axisbelow(True)
axes[0].set_ylabel("paired HSR gap to dense"); axes[0].legend(frameon=False, loc="upper left", ncol=2, handlelength=1.2, columnspacing=0.6)
fig.savefig(OUT / "N45_headroom.png", dpi=600, bbox_inches="tight"); fig.savefig(OUT / "N45_headroom.pdf", bbox_inches="tight"); plt.show()

# figure 2: per-family recall at 90%, cheap pass and two epochs, dense + five rules, both networks
fig, axes = plt.subplots(1, 4, figsize=(7.16, 1.6), gridspec_kw={"wspace": 0.12})
order = ["dense"] + METHODS; FAM_ORDER = [f for f in ["benign", "ddos", "dos", "mqtt", "recon", "spoofing"] if f in FAMILIES] + [f for f in FAMILIES if f not in ["benign", "ddos", "dos", "mqtt", "recon", "spoofing"]]
for ax, (arch, reg) in zip(axes, [(a, r) for a in ARCHS for r in ("minimal", "standard")]):
    mat = np.array([[float(SUM[(SUM.regime == reg) & (SUM.architecture == arch) & (SUM.method == o) & (np.isclose(SUM.budget, 0.90) | (o == "dense"))][f"recal_recall_{f}"].iloc[0]) for f in FAM_ORDER] for o in order])
    ax.imshow(mat, vmin=0, vmax=1, cmap="Blues", aspect="auto")
    for i in range(len(order)):
        for j in range(len(FAM_ORDER)):
            v = mat[i, j]; ax.text(j, i, f"{v:.2f}".lstrip("0") if v < 0.995 else "1.0", ha="center", va="center", fontsize=5, color="white" if v > 0.6 else "#1F1F1F")
    ax.set_xticks(range(len(FAM_ORDER))); ax.set_xticklabels([FAMN[f] for f in FAM_ORDER], fontsize=5.5, rotation=90)
    ax.set_yticks(range(len(order))); ax.set_yticklabels([NAME[o] for o in order], fontsize=6)
    if ax is not axes[0]: ax.set_yticklabels([])
    ax.set_title(f"{arch}, {'cheap pass' if reg == 'minimal' else 'two epochs'}", fontsize=7, pad=2); ax.tick_params(length=0)
fig.savefig(OUT / "N45_families.png", dpi=600, bbox_inches="tight"); fig.savefig(OUT / "N45_families.pdf", bbox_inches="tight"); plt.show()

# figure 3: ECE of dense reference and students by budget (balanced subsample), both networks, cheap pass; two-epoch points at right
fig, axes = plt.subplots(1, 2, figsize=(3.5, 1.8), sharey=True, gridspec_kw={"wspace": 0.08})
for ax, arch in zip(axes, ARCHS):
    for reg, x_dense, mfc in (("minimal", 40, None), ("standard", 97.5, "white")):
        d = SUM[(SUM.regime == reg) & (SUM.architecture == arch) & (SUM.method == "dense")].recal_ece.iloc[0]
        ax.axhline(d, color=COLR["dense"], lw=0.8, ls="-" if reg == "minimal" else "--", zorder=1)
    for k, m in enumerate(METHODS):
        g = SUM[(SUM.regime == "minimal") & (SUM.architecture == arch) & (SUM.method == m)].sort_values("budget")
        ax.plot(100 * g.budget + (k - 2) * 0.5, g.recal_ece, color=COLR[m], marker=MARK[m], ms=2.6, lw=0.8, label=NAME[m], zorder=3)
        f = SUM[(SUM.regime == "standard") & (SUM.architecture == arch) & (SUM.method == m)]
        if len(f): ax.scatter([97.5 + (k - 2) * 0.5], f.recal_ece, color=COLR[m], marker=MARK[m], s=12, facecolor="white", zorder=3)
    ax.set_title(f"{arch} network", fontsize=7, pad=2); ax.set_xticks([40, 70, 80, 90, 97.5]); ax.set_xticklabels(["40", "70", "80", "90", "90\ntwo ep."]); ax.set_xlabel("realised reduction (%)")
    ax.spines[["top", "right"]].set_visible(False); ax.grid(axis="y", color="#E6E6E6", lw=0.5); ax.set_axisbelow(True)
axes[0].set_ylabel("ECE (dense: lines)"); axes[0].legend(frameon=False, loc="best", ncol=2, handlelength=1.2, columnspacing=0.6)
fig.savefig(OUT / "N45_calibration.png", dpi=600, bbox_inches="tight"); fig.savefig(OUT / "N45_calibration.pdf", bbox_inches="tight"); plt.show()
print("written ->", OUT)

In [ ]:
# Stage 9 - commit the results (standard commit cell; the message is filled in from the verdict)
v = json.load(open(OUT / "N45_verdict.json"))
pre = subprocess.run(["git", "log", "-1", "--format=%h", "--", "results/saber/45_iomt_headroom/N45_PREREGISTRATION.json"], capture_output=True, text=True, cwd=REPO).stdout.strip() or "see log"
parts = []
for arch in ARCHS:
    h = v["by_architecture"][arch]; b90 = h["H1_by_budget"]["0.90"]; f90 = h["full_90"]; c = h["C_calibration"]
    lost = {NAME[m]: fams for m, fams in h["F_lost_at_90_cheap"].items() if fams}
    parts.append(f"{arch}: cheap-pass headroom {'opens at ' + format(h['H1_cheap_knee'], '.2f') if h['H1_cheap_knee'] else 'none'} "
                 f"(90% best {NAME[b90['best_rule']]} gap {b90['gap']:+.3f} [{b90['gap_lo']:+.3f}, {b90['gap_hi']:+.3f}]); two epochs at 90%: "
                 f"{'HEADROOM' if h['H1_full_90'] else 'no headroom'} (best {NAME[f90['best_rule']]} gap {f90['gap']:+.3f} [{f90['gap_lo']:+.3f}, {f90['gap_hi']:+.3f}]); "
                 f"E at 90% cheap {h['E_range_90_cheap'][0]:.3f}-{h['E_range_90_cheap'][1]:.3f} vs dense {h['E_dense_cheap']:.3f}; families lost under the cheap pass "
                 f"{lost if lost else 'none'}, repaired by two epochs (standard rules) {h['F_repaired_by_two_epochs_standard_rules']}; calibration: student better in "
                 f"{c['student_better']}, worse in {c['student_worse']} of {c['cells']} cells (dense ECE {c['dense_ece_balanced']['standard']:.3f} balanced / {c['dense_ece_natural']['standard']:.3f} natural prior at two epochs).")
X = v["cross_corpus_expectations"]
msg = (f"NB45 results (pre-registration {pre}): compression headroom on CIC-IoMT-2024, {v['runs']} recovered models. " + " ".join(parts)
       + f" Cross-corpus expectations: X1 {'HELD' if X['X1_no_headroom_at_90_after_two_epochs_both'] else 'NOT HELD'}, "
       f"X2 {'HELD' if X['X2_students_better_calibrated_in_at_least_half'] else 'NOT HELD'} ({v['calibration_all_cells']['student_better']} better / {v['calibration_all_cells']['student_worse']} worse of {v['calibration_all_cells']['cells']}), "
       f"X3 {'HELD' if X['X3_cheap_pass_loses_a_family_on_some_network'] else 'NOT HELD'}. As-trained collapses by the gap rule: {v['collapses_as_trained']}.")
print(msg)
os.chdir(REPO)
r = subprocess.run(["python", "tools/commit_cell.py", msg], capture_output=True, text=True)
print(r.stdout); print(r.stderr)